In [ ]:
pip install pdfplumber PyMuPDF pytesseract pdf2image pillow docling camelot chromadb

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

#!rm -rf /content/drive/MyDrive/Colab_Chatbot/chroma_db
#!rm -rf /content/drive/MyDrive/Colab_Chatbot/saved_model

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import chromadb
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Extracted_folder = "/content/drive/MyDrive/Colab_Chatbot/Extracted CSV"

#Presentation_folder = "drive/MyDrive/Project File/Embedded CSV"

i=0
for file in os.listdir(Extracted_folder):
    i = i + 1
    if file.endswith(".csv"):  # Process only .txt files
        file_path = os.path.join(Extracted_folder, file)
        pdf_name = os.path.splitext(file)[0]

        print(f"Processing: {i} {pdf_name}")

            # Read extracted text
        df = pd.read_csv(file_path)

        model = SentenceTransformer("all-MiniLM-L12-v2")

            # Initialize ChromaDB client
        chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Colab_Chatbot/chroma_db")

            # Create a new collection
        collection = chroma_client.get_or_create_collection(name="ebrr_analysis",metadata={"hnsw:space": "cosine"})

        # Extract data
        documents = df["chunk_text"].tolist()
        metadatas = df[["filename", "chunk_id"]].to_dict(orient="records")
        ids = [f"{row['filename']}_{row['chunk_id']}" for _, row in df.iterrows()]

        # Generate embeddings
        embeddings = model.encode(documents).tolist()
        try:
          collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
          )
        except ValueError:
            print("Empty file input.")


        print("Data stored in ChromaDB successfully!")


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Colab_Chatbot/chroma_db")
collection = chroma_client.get_or_create_collection(name="ebrr_analysis",metadata={"hnsw:space": "cosine"})

collection.count()

In [ ]:
!pip install streamlit==1.44.1 PyPDF2==3.0.1 joblib==1.4.2 numpy==2.0.2 pandas==2.2.2 torch==2.6.0 datasets==3.5.0 pyngrok==7.2.3 fsspec==2024.12.0 gcsfs==2024.12.0 import-ipynb

In [ ]:
# Streamlit_Colab.ipynb

# Install dependencies with compatible versions

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create a directory in Google Drive for persistent storage
import os
if not os.path.exists('/content/drive/MyDrive/Colab_Chatbot'):
    os.makedirs('/content/drive/MyDrive/Colab_Chatbot')
if not os.path.exists('/content/drive/MyDrive/Colab_Chatbot/chroma_db'):
    os.makedirs('/content/drive/MyDrive/Colab_Chatbot/chroma_db')
if not os.path.exists('/content/drive/MyDrive/Colab_Chatbot/saved_model'):
    os.makedirs('/content/drive/MyDrive/Colab_Chatbot/saved_model')

Extracted_code = r"""
import fitz
import cv2
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from pdf2image.exceptions import PDFPageCountError
from sentence_transformers import SentenceTransformer
from PIL import Image
import os
import pandas as pd
import textwrap
import re
import chromadb

# Set your PDF path
folder_path = "drive/MyDrive/Colab Notebooks/Raw Developer 2"
filename = os.path.basename(folder_path)

output_folder = "drive/MyDrive/Project File/Extracted CSV"
os.makedirs(output_folder, exist_ok=True)

pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

# Universal cleaner function
def clean_text_universal(text):
    text = re.sub(r'--- Page \d+ ---', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\S+|doi\S+', ' ', text)
    text = re.sub(r'(\w+)-\s*\\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_text(text):
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove multiple spaces, new lines
    text = re.sub(r'[^\w\s.,%-]', '', text)  # Remove special characters
    return clean_text_universal(text.lower())

#  Chunking function
def split_text_into_chunks(text, chunk_size=300):
    paragraphs = text.split("\\n\\n")
    chunks = []
    chunk_id = 0
    for para in paragraphs:
        if len(para.strip()) < 100:
            continue
        wrapped_chunks = textwrap.wrap(para.strip(), chunk_size)
        for chunk in wrapped_chunks:
            chunk_id += 1
            chunks.append((f"P1-C{chunk_id}", chunk))
    return chunks

#  First try pdfplumber
def extract_text_pdfplumber(pdf_path):
    full_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    full_text += f"\\n\\n--- Page {i+1} ---\\n{text}"
    except:
        pass
    return full_text.strip()

#  OCR fallback for scanned PDFs
def extract_text_ocr(pdf_path,images):
    print("[INFO] Using OCR fallback for scanned PDF...")
    extracted_text = []
    for i, img in enumerate(images):
        img_path = os.path.join(output_folder, f"page_{i+1}.png")
        img.save(img_path, "PNG")

        # Read the saved image using OpenCV
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Apply Adaptive Thresholding
        processed_img = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                              cv2.THRESH_BINARY, 11, 2)
        processed_img = cv2.GaussianBlur(processed_img, (5, 5), 0)

        # Save processed image
        processed_img_path = os.path.join(output_folder, f"processed_page_{i+1}.png")
        cv2.imwrite(processed_img_path, processed_img)

        # Perform OCR
        text = pytesseract.image_to_string(processed_img, config='--psm 6')
        extracted_text.append(text)

    return "\\n".join(extracted_text)

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\\n"
    return text


#  Universal text extractor
def extract_text_universal(pdf_path):
	text = extract_text_pdfplumber(pdf_path)
	if len(text) < 100:
		text = extract_text_from_pdf(pdf_path)
	if len(text) < 100:  # not enough text, fallback to OCR
		try:
		  images = convert_from_path(pdf_path,dpi=300)
		  text = extract_text_ocr(pdf_path, images)
		except PDFPageCountError as e:
			print(f"Skipping {pdf_path} due to error: {e}")
		except Exception as e:
			print(f"Unexpected error for {pdf_path}: {e}")
	return clean_text(text)


def pdf_extraction(file):
    full_text = extract_text_universal(file)
    chunks = split_text_into_chunks(full_text)
    df = pd.DataFrame(chunks, columns=["chunk_id", "chunk_text"])
    filename = os.path.basename(file)
    df.insert(0, "filename", filename)

    chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Colab_Chatbot/chroma_db")

    # Create a new collection
    collection = chroma_client.get_or_create_collection(name="ebrr_analysis",metadata={"hnsw:space": "cosine"})

    model = SentenceTransformer("all-MiniLM-L12-v2")

    # Extract data
    documents = df["chunk_text"].tolist()
    metadatas = df[["filename", "chunk_id"]].to_dict(orient="records")
    ids = [f"{row['filename']}_{row['chunk_id']}" for _, row in df.iterrows()]

    # Generate embeddings
    embeddings = model.encode(documents).tolist()
    try:
		    collection.add(
		      documents=documents,
          embeddings=embeddings,
		      metadatas=metadatas,
		      ids=ids
		    )
    except ValueError:
    	  return "Empty file input."

    return filename

"""

with open("extracted.py", "w", encoding="utf-8") as fs:
    fs.write(Extracted_code)

# Write the backend code to a file (needed for import in Streamlit)
backend_code = """
from transformers import AutoTokenizer, AutoModelForCausalLM, BertTokenizer, BertForSequenceClassification
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from datasets import Dataset
from transformers import Trainer, TrainingArguments
import joblib
import torch
import os

# Initialize ChromaDB with persistent storage in Google Drive
chroma_client = PersistentClient(path="/content/drive/MyDrive/Colab_Chatbot/chroma_db")
collection = chroma_client.get_or_create_collection(name="ebrr_analysis",metadata={"hnsw:space": "cosine"})

def save_models():
    try:
        if not os.path.exists("/content/drive/MyDrive/Colab_Chatbot/saved_model"):
            os.makedirs("/content/drive/MyDrive/Colab_Chatbot/saved_model")
        print("Saving generative model (gpt2)...")
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
        model.save_pretrained("/content/drive/MyDrive/Colab_Chatbot/saved_model/bert", safe_serialization=True)
        tokenizer.save_pretrained("/content/drive/MyDrive/Colab_Chatbot/saved_model/bert")
        print("Saving embedder...")
        embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
        joblib.dump(embedder, "/content/drive/MyDrive/Colab_Chatbot/saved_model/embedder.pkl")
        print("Models saved successfully to Google Drive.")
    except Exception as e:
        print(f"Error saving models: {str(e)}")
        raise

def load_models():
    try:
        print("Loading tokenizer and model...")
        tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/Colab_Chatbot/saved_model/bert")
        model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/Colab_Chatbot/saved_model/bert")
        print("Loading embedder...")
        embedder = joblib.load("/content/drive/MyDrive/Colab_Chatbot/saved_model/embedder.pkl")

        # Initialize ChromaDB collection with HNSW indexing
        client = PersistentClient(path="/content/drive/MyDrive/Colab_Chatbot/chroma_db")
        collection = client.get_or_create_collection(
            name="cbt_analysis",
            metadata={"hnsw:space": "cosine"}  # or "l2" if you want L2 norm
        )

        return tokenizer, model, embedder
    except Exception as e:
        print(f"Error loading models: {str(e)}")
        return None, None, None

def add_program_data(doc, program_name, recidivism_reduction):
    embedder = load_models()[2]
    if embedder:
        embedding = embedder.encode(doc).tolist()
        collection.add(
            embeddings=[embedding],
            documents=[doc],
            metadatas=[{"program_name": program_name, "recidivism_reduction": recidivism_reduction}],
            ids=[f"prog_{program_name.lower().replace(' ', '_')}"]
        )

def get_response(query, filename_filter=None, selected_number = 3):
    tokenizer, model, embedder = load_models()
    if not all([tokenizer, model, embedder]):
        return "Error: Models not loaded."
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=5, where={"filename": filename_filter} if filename_filter else None)
    top_docs = results["documents"][0]  # Assuming structure is [ [doc1, doc2, ...] ]
    top_scores = results["distances"][0]  # Lower is better for distance
    top_metadatas = results["metadatas"][0]

    # Pair them up and sort by relevance
    doc_score_pairs = sorted(zip(top_docs, top_scores, top_metadatas), key=lambda x: x[1])
    top_k = selected_number  # You can tune this
    selected_context = " ".join([doc for doc, _, _ in doc_score_pairs[:top_k]])
    scores_top_k = [1 / (1 + score) if score > 1 else 1.0 - score for _, score, _ in doc_score_pairs[:top_k]]

    filenames_top_k = [meta.get("filename", "Unknown") for _, _, meta in doc_score_pairs[:top_k]]

    return selected_context, scores_top_k, filenames_top_k

def T5_text_generation(retrieved_text):
    # Load T5 model
    t5_tokenizer = T5Tokenizer.from_pretrained("t5-base")
    t5_model = T5ForConditionalGeneration.from_pretrained("t5-base")

    # Prepare input for T5
    prompt = f"summarize: {retrieved_text}"

    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=512)
    output = t5_model.generate(**inputs, max_length=350, num_beams=4, early_stopping=True)
    generated_text = t5_tokenizer.decode(output[0], skip_special_tokens=True)

    return generated_text
"""

with open("backend.py", "w") as f:
    f.write(backend_code)

#prompt = f"Based on this data: {context} \\n \\n Answer: {query}"
#    inputs = tokenizer(prompt, return_tensors="pt")
#    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True)

# Write the Streamlit app code to a file
streamlit_code = """
import streamlit as st
from backend import get_response, add_program_data, load_models, T5_text_generation, save_models
from extracted import pdf_extraction
from chromadb import PersistentClient
import PyPDF2
import tempfile
import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
import re
import os


# Main Streamlit App
def main():
    st.title("EBRR Program Validation Platform")

    if "qa_pairs" not in st.session_state:
        st.session_state.qa_pairs = []  # stores (question, selected_filename, response)
    if "ask_more" not in st.session_state:
        st.session_state.ask_more = True

    # Initialize ChromaDB with persistent storage
    chroma_client = PersistentClient(path="/content/drive/MyDrive/Colab_Chatbot/chroma_db")
    collection = chroma_client.get_or_create_collection(name="ebrr_analysis",metadata={"hnsw:space": "cosine"})

    # Load models from backend
    if not os.path.exists("/content/drive/MyDrive/Colab_Chatbot/saved_model/bert"):
            save_models()
    tokenizer, model, embedder = load_models()
    print(tokenizer, model, embedder)
    if not all([tokenizer, model, embedder]):
        st.error("Failed to initialize models. Please ensure models are saved in '/content/drive/MyDrive/Colab_Chatbot/saved_model/'.")
        return

    st.session_state.tokenizer = tokenizer
    st.session_state.model = model
    st.session_state.embedder = embedder

    # Display current collection size
    st.write(f"Current database size: {collection.count()} documents")

    # Tabs
    # tab1, tab2, tab3 = st.tabs(["Chatbot", "PDF Upload", "Meta-Analysis"])

    tab1, tab2, tab3 = st.tabs(["Chatbot", "PDF Upload", "Manage Files"])

    # Chatbot Tab
    with tab1:
        st.subheader("Chatbot")

        if st.session_state.ask_more:
            # --- Get file options from metadata
            all_metadatas = collection.get(include=["metadatas"])["metadatas"]
            filenames = sorted(list({meta.get("filename", "Unknown") for meta in all_metadatas}))
            filenames.insert(0, "All files")  # default option

            # --- Input boxes
            col1, col2 = st.columns([2, 1])

            with col1:
                selected_file = st.selectbox("Select a file to query from:", filenames)
            with col2:
                selected_number = st.selectbox("Select Top k number of documents:", list(range(1, 6)), index=2)
            question = st.text_input("Your question:")

            if st.button("Submit Question"):
              if question.strip():
                  filename_filter = None if selected_file == "All files" else selected_file
                  response, scores_top_k, filenames_top_k = get_response(question, filename_filter, selected_number)

                  avg_score = sum(scores_top_k) / len(scores_top_k) if scores_top_k else 0

                  if avg_score < 0.3:
                    st.warning("The answer cannot be determined from the available files based on the current data.")
                    st.session_state.qa_pairs.append((question, selected_file, "No reliable answer determined from the available files based on the current data.", None))
                  else:
                    text_generation = T5_text_generation(response)

                    df_scores = pd.DataFrame({
                        'Result Documents': [f"Result {i+1}" for i in range(len(scores_top_k))],
                        'file': filenames_top_k,
                        'Similarity Score': scores_top_k
                    })

                    chart = alt.Chart(df_scores).mark_bar().encode(
                      x=alt.X('Result Documents:N', sort=None),
                      y='Similarity Score:Q',
                      color=alt.Color('Similarity Score:Q', scale=alt.Scale(scheme='blues')),
                      tooltip=['Result Documents', 'file', 'Similarity Score']
                    ).properties(height=300, title=f"Top {selected_number} Similarity Scores:")

                    st.session_state.qa_pairs.append((question, selected_file, text_generation, chart))
              else:
                st.warning("Please enter a question.")

            if st.button("Done Asking Questions"):
                st.session_state.ask_more = False



        # --- Display all past Q&A
        if st.session_state.qa_pairs:
            st.markdown("### Q&A History")
            for i, (q, f, r, p) in enumerate(st.session_state.qa_pairs, 1):
                st.markdown(f"**Q{i}:** *{q}* (from `{f}`)")
                st.markdown(f"**Response:** {r}")
                if p is not None:
                  st.altair_chart(p, use_container_width=True)
                st.markdown("---")

        # --- Reset
        if not st.session_state.ask_more:
            if st.button("Start Over"):
                st.session_state.qa_pairs = []
                st.session_state.ask_more = True

    # PDF Upload Tab
    with tab2:
        st.subheader("Upload PDF")
        uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])

        if uploaded_file is not None:
            st.success("File uploaded successfully.")

              # Write to a temp file for pdfplumber (if needed)
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
                tmp_file.write(uploaded_file.getbuffer())
                tmp_path = tmp_file.name
                if tmp_path:
                  num_programs = pdf_extraction(tmp_path)
                  if num_programs:
                    st.success(f"Extracted {num_programs} programs from {uploaded_file.name}. Added to database.")
                  else:
                    st.warning("No programs extracted from the PDF. Added text chunks to database.")
    with tab3:
        st.subheader("Delete Documents by Filename")

        # Fetch metadata and corresponding IDs
        all_data = collection.get(include=["metadatas"])
        all_metadatas = all_data["metadatas"]
        all_ids = all_data["ids"]

        filenames = sorted(list({meta.get("filename", "Unknown") for meta in all_metadatas}))
        selected_files = st.multiselect("Select filenames to delete:", filenames)

        if selected_files:
            # Find matching IDs
            matching_ids = [
                doc_id for doc_id, meta in zip(all_ids, all_metadatas)
                if meta.get("filename") in selected_files
            ]

            st.info(f"Found {len(matching_ids)} documents matching selected filenames.")

            if st.button("Delete Selected Documents"):
                collection.delete(ids=matching_ids)
                st.success(f"Deleted {len(matching_ids)} documents.")
        else:
            st.warning("Select at least one filename to delete.")




if __name__ == "__main__":
    main()
"""

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

# Set up ngrok for Streamlit
import os
import time
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()

# Set up ngrok with your authtoken (get it from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "2v9YiLf3h94F9Qlm5QCth0AR4M8_2E7MGuupNWLpFEAF3VKVu"  # Replace with your ngrok authtoken
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Run Streamlit in background
os.system("nohup streamlit run streamlit_app.py &")

# Wait a moment for the server to start
time.sleep(3)

# Create public tunnel
public_url = ngrok.connect(addr=8501)
print(f"✅ Streamlit app is live at: {public_url}")